# 실습 ② — 불량 사진 없이 검사하기 (이상탐지)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abh-edu/mfg-ai-course/blob/main/lab2_vision_anomaly.ipynb)

**제조 AI 심화과정 · 5~6교시** — 정상 사진만 외워서, 처음 보는 불량을 찾아냅니다.

- 칸 왼쪽의 ▶ 를 **위에서 아래로 차례로** 누릅니다. 코드는 읽지 않으셔도 됩니다.
- 빨간 글씨(Warning)는 대부분 정상입니다. **Error** 가 보이면 그 칸을 한 번 더 실행해 보세요.
- GPU는 필요 없습니다. 쉬는 시간에 돌려 두는 한 칸(5분쯤)을 빼면, 가장 오래 걸리는 칸이 1분 남짓입니다.

데이터: VisA (Zou et al., ECCV 2022, Amazon) · CC BY 4.0 · 수업용으로 줄인 발췌본


In [ ]:
#@title STEP 0 · 도구 준비 — 이 칸은 누르기만 하면 됩니다 (약 30초)
import os, time, zipfile, urllib.request, warnings
import numpy as np, pandas as pd, torch, torchvision
import torch.nn.functional as F
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
from PIL import Image
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

# 그래프에 한글이 보이도록 글꼴 준비 (실패해도 실습에는 지장이 없습니다)
try:
    os.system('pip install -q koreanize-matplotlib'); import koreanize_matplotlib
except Exception:
    print('한글 글꼴을 준비하지 못했습니다. 그래프의 글자가 깨져 보일 수 있습니다.')
plt.rc('axes', unicode_minus=False); plt.rc('figure', dpi=110)
%config InlineBackend.figure_format = 'jpeg'
%config InlineBackend.print_figure_kwargs = {'bbox_inches': 'tight', 'pil_kwargs': {'quality': 88}}

주소 = 'https://raw.githubusercontent.com/abh-edu/mfg-ai-course/main/'
장치 = 'cuda' if torch.cuda.is_available() else 'cpu'
청록, 주황, 남색, 회색 = '#1798AE', '#EE5A24', '#1D2053', '#5F6B7A'

# 이미 사물을 볼 줄 아는 공개 모델(ResNet-18)의 "눈"을 빌립니다 — 새로 학습시키지 않습니다
눈 = torchvision.models.resnet18(weights='DEFAULT').eval().to(장치); _f = {}
눈.layer2.register_forward_hook(lambda m, i, o: _f.__setitem__('a', o))
눈.layer3.register_forward_hook(lambda m, i, o: _f.__setitem__('b', o))
_평균 = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); _편차 = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
torch.manual_seed(0); _투영 = (torch.randn(384, 128) / 128 ** 0.5).to(장치)

def 불러오기(제품):
    """제품 사진 묶음을 내려받아 용도별 목록으로 나눕니다"""
    if not os.path.isdir('data/' + 제품):
        os.makedirs('data', exist_ok=True); zp = 'data/visa_' + 제품 + '.zip'
        urllib.request.urlretrieve(주소 + 'data/visa_' + 제품 + '.zip', zp); zipfile.ZipFile(zp).extractall('data')
    df = pd.read_csv('data/' + 제품 + '/index.csv'); P = lambda s: ['data/' + p for p in s]
    정상 = P(df[df.split == 'train'].image); 시험 = df[df.split == 'test']; 불량 = 시험[시험.label == 'bad']
    자료 = dict(제품=제품, 기억=정상[:120], 기준=정상[120:], 시험_정상=P(시험[시험.label == 'good'].image), 시험_불량=P(불량.image), 정답위치=P(불량['mask']), 불량종류=list(불량.defect))
    print(f"준비 완료 [{제품}] 외울 정상 {len(자료['기억'])}장 · 기준선용 정상 {len(자료['기준'])}장 · 시험용 정상 {len(자료['시험_정상'])}장 · 시험용 불량 {len(자료['시험_불량'])}장")
    return 자료

@torch.no_grad()
def 조각_특징(경로들, 해상도=256):
    """사진을 작은 조각으로 나눠, 조각마다 특징(숫자 128개)을 뽑습니다"""
    out = []
    for i in range(0, len(경로들), 16):
        x = torch.stack([torch.from_numpy(np.asarray(Image.open(p).convert('RGB').resize((해상도, 해상도), Image.BILINEAR), dtype=np.float32) / 255.).permute(2, 0, 1) for p in 경로들[i:i + 16]])
        눈(((x - _평균) / _편차).to(장치))
        a = F.avg_pool2d(_f['a'], 3, 1, 1); b = F.interpolate(F.avg_pool2d(_f['b'], 3, 1, 1), size=a.shape[-2:], mode='bilinear', align_corners=False)
        out.append((torch.cat([a, b], 1).permute(0, 2, 3, 1) @ _투영).cpu())
    return torch.cat(out)

def 외우기(경로들, 해상도=256, 창고_크기=20000):
    """정상 사진의 조각 특징을 기억 창고에 넣습니다 (전부는 많으니 2만 개만 무작위로)"""
    e = 조각_특징(경로들, 해상도); 전체 = e.reshape(-1, e.shape[-1])
    고른 = np.random.RandomState(0).choice(len(전체), min(창고_크기, len(전체)), replace=False)
    return 전체[torch.from_numpy(고른)], len(전체)

@torch.no_grad()
def 거리_지도(창고, 경로들, 해상도=256):
    """조각마다 '창고에서 가장 닮은 조각까지의 거리'를 잽니다 — 멀수록 이상합니다"""
    e = 조각_특징(경로들, 해상도); N, H, W, C = e.shape; b = 창고.to(장치)
    return torch.stack([torch.cdist(e[i].reshape(-1, C).to(장치), b).min(1).values.reshape(H, W).cpu() for i in range(N)])

def 이상_점수(지도):
    """사진 한 장의 점수 = 가장 이상한 조각 1%의 평균 거리"""
    k = max(1, 지도[0].numel() // 100)
    return 지도.flatten(1).topk(k, dim=1).values.mean(1).numpy()

def 구분력(정상, 불량):
    """정상과 불량을 한 장씩 뽑았을 때 불량의 점수가 더 높을 확률(%) — AUROC"""
    return roc_auc_score(np.r_[np.zeros(len(정상)), np.ones(len(불량))], np.r_[정상, 불량]) * 100

def 히트맵(지도_한장, 크기):
    return F.interpolate(F.avg_pool2d(지도_한장[None, None], 3, 1, 1), size=크기, mode='bilinear', align_corners=False)[0, 0].numpy()

def 눈금(기준_지도):
    """히트맵 색의 기준 — 정상 사진에서 나온 가장 높은 값. 이 값을 넘는 곳부터 색을 칠합니다"""
    return float(F.avg_pool2d(기준_지도[:, None], 3, 1, 1).max())

def 위치_적중(자료, 지도):
    """모델이 가장 이상하다고 짚은 지점이 실제 결함 위치(정답) 근처에 드는지 채점합니다"""
    맞음 = []
    for i, mp in enumerate(자료['정답위치']):
        m = np.asarray(Image.open(mp).convert('L')) > 0; H, W = m.shape; h = 히트맵(지도[i], (H, W)); y, x = np.unravel_index(h.argmax(), h.shape)
        r = max(H, W) // 20; 맞음.append(bool(m[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1].any()))
    return np.array(맞음)

_말 = [('missing', '누락'), ('burnt', '탄 자국'), ('scratch', '긁힘'), ('dirt', '오염'), ('damage', '손상'), ('extra', '이물'), ('wrong place', '위치 틀어짐'), ('melt', '녹음'), ('bent', '휨'), ('breakage', '깨짐'), ('colour spot', '얼룩'), ('crack', '균열'), ('hole', '구멍'), ('stuck', '붙음'), ('bubble', '기포'), ('discolor', '변색'), ('leak', '샘'), ('misshape', '모양 불량'), ('wick', '심지 불량'), ('partical', '이물'), ('wax', '왁스 불량')]
def 우리말(s):
    out = [v for k, v in _말 if k in str(s).lower()]
    return ' · '.join(dict.fromkeys(out)) if out else str(s)

def 문제_고르기(자료):
    """결함이 큰 것부터 작은 것까지 불량 5장 + 정상 3장을 골라 섞습니다"""
    면적 = [np.asarray(Image.open(m).convert('L')).mean() for m in 자료['정답위치']]
    순서 = np.argsort(면적)[::-1]; n = len(순서)
    문제 = [dict(사진=자료['시험_불량'][i], 불량=True, 위치=자료['정답위치'][i], 종류=자료['불량종류'][i]) for i in [int(순서[int(n * q)]) for q in (0.05, 0.25, 0.5, 0.7, 0.9)]]
    문제 += [dict(사진=p, 불량=False) for p in 자료['시험_정상'][:3]]
    np.random.RandomState(7).shuffle(문제); return 문제

def 보여주기(문제, 정답=False):
    fig, axes = plt.subplots(2, 4, figsize=(16, 7.6))
    for k, (ax, q) in enumerate(zip(axes.ravel(), 문제)):
        ax.imshow(Image.open(q['사진']).convert('RGB')); ax.axis('off')
        if 정답 and q['불량']:
            ax.contour((np.asarray(Image.open(q['위치']).convert('L')) > 0).astype(float), levels=[0.5], colors='red', linewidths=2.5)
            ax.set_title(str(k + 1) + '번  불량 — ' + 우리말(q['종류']), color=주황, fontsize=15, fontweight='bold')
        elif 정답: ax.set_title(str(k + 1) + '번  정상', color=청록, fontsize=15, fontweight='bold')
        else: ax.set_title(str(k + 1) + '번', color=남색, fontsize=15, fontweight='bold')
    plt.tight_layout(); plt.show()

def 점수분포_그림(정상, 불량, 기준선, 제목=''):
    lo, hi = min(정상.min(), 불량.min()), max(정상.max(), 불량.max()); bins = np.linspace(lo, hi, 31)
    plt.figure(figsize=(11, 4.2))
    plt.hist(정상, bins, color=청록, alpha=.85, label='정상 ' + str(len(정상)) + '장'); plt.hist(불량, bins, color=주황, alpha=.7, label='불량 ' + str(len(불량)) + '장')
    plt.axvline(기준선, color=남색, ls='--', lw=2.5, label='기준선'); plt.xlabel('이상 점수 (클수록 정상에서 멀다)'); plt.ylabel('사진 수'); plt.legend(fontsize=12)
    plt.title(제목 or '점수 분포 — 정상과 불량이 갈라집니까?', fontsize=14, fontweight='bold'); plt.show()

def 판정표(정상, 불량, 기준선, 그림=True):
    미검 = int((불량 <= 기준선).sum()); 과검 = int((정상 > 기준선).sum()); TP = len(불량) - 미검; TN = len(정상) - 과검
    print(f'불량 {len(불량)}장 중 {TP}장 검출 · 미검(놓침) {미검}장  |  정상 {len(정상)}장 중 과검(헛경보) {과검}장')
    if 그림:
        fig, ax = plt.subplots(figsize=(5.4, 4.6)); 값 = [[TN, 과검], [미검, TP]]; 색 = [['#F4F6F8', '#FFF6D6'], ['#FDECE4', '#E3F3F6']]; 글 = [['맞게 통과', '과검 (헛경보)'], ['미검 (놓침)', '맞게 검출']]
        for r in range(2):
            for c in range(2):
                ax.add_patch(plt.Rectangle((c, 1 - r), 1, 1, fc=색[r][c], ec='white', lw=4))
                ax.text(c + .5, 1.58 - r, str(값[r][c]), ha='center', va='center', fontsize=30, fontweight='bold', color=주황 if (r, c) == (1, 0) else 남색); ax.text(c + .5, 1.2 - r, 글[r][c], ha='center', fontsize=12, color=회색)
        ax.set_xlim(0, 2); ax.set_ylim(0, 2); ax.set_xticks([.5, 1.5]); ax.set_xticklabels(['정상이라 판정', '불량이라 판정'], fontsize=12); ax.set_yticks([.5, 1.5]); ax.set_yticklabels(['실제 불량', '실제 정상'], fontsize=12)
        ax.xaxis.tick_top(); ax.tick_params(length=0)
        for sp in ax.spines.values(): sp.set_visible(False)
        plt.show()
    return dict(TN=TN, FP=과검, FN=미검, TP=TP)

def _칠하기(ax, im, h, 기준):
    hi = max(float(h.max()), 기준 * 1.05); a = np.clip((h - 기준) / (hi - 기준), 0, 1); 색 = plt.cm.jet(a); 색[..., 3] = np.where(a > 0, 0.25 + a * 0.6, 0)
    ax.imshow(im); ax.imshow(색); ax.axis('off')

def 히트맵_보기(자료, 지도, 번호, 범위):
    번호 = list(번호); n = len(번호); fig, axes = plt.subplots(3, n, figsize=(4.2 * n, 9), squeeze=False)
    for j, i in enumerate(번호):
        im = Image.open(자료['시험_불량'][i]).convert('RGB'); W, H = im.size
        h = 히트맵(지도[i], (H, W)); m = np.asarray(Image.open(자료['정답위치'][i]).convert('L')) > 0; y, x = np.unravel_index(h.argmax(), h.shape)
        axes[0, j].imshow(im); axes[0, j].axis('off'); axes[0, j].set_title('원본 — ' + 우리말(자료['불량종류'][i]), fontsize=12, color=남색, fontweight='bold')
        _칠하기(axes[1, j], im, h, 범위); axes[1, j].set_title('모델이 이상하다고 본 곳', fontsize=12, color=남색)
        axes[2, j].imshow(im); axes[2, j].contour(m.astype(float), levels=[0.5], colors='red', linewidths=2.2); axes[2, j].plot([x], [y], marker='+', color='cyan', markersize=22, markeredgewidth=3.5)
        axes[2, j].axis('off'); axes[2, j].set_title('정답 위치(빨간 선)와 모델이 짚은 곳(+)', fontsize=12, color=남색)
    plt.tight_layout(); plt.show()

def 과검_보기(자료, 지도, 번호, 범위):
    번호 = list(번호); n = len(번호); fig, axes = plt.subplots(2, n, figsize=(4.2 * n, 6), squeeze=False)
    for j, i in enumerate(번호):
        im = Image.open(자료['시험_정상'][i]).convert('RGB'); W, H = im.size; h = 히트맵(지도[i], (H, W))
        axes[0, j].imshow(im); axes[0, j].axis('off'); axes[0, j].set_title('정상인데 불량으로 판정', fontsize=12, color=남색, fontweight='bold')
        _칠하기(axes[1, j], im, h, 범위); axes[1, j].set_title('모델이 이상하다고 본 곳', fontsize=12, color=남색)
    plt.tight_layout(); plt.show()

def 한번에(제품, 기억할_장수=120, 해상도=256, 그림=True):
    """STEP 2~5를 한 번에 — 제품 이름만 바꿔 다시 해 볼 때 씁니다"""
    t = time.time(); 자료 = 불러오기(제품); 창고, _ = 외우기(자료['기억'][:기억할_장수], 해상도)
    기준_지도 = 거리_지도(창고, 자료['기준'], 해상도); 기준선 = float(이상_점수(기준_지도).max())
    정상_지도 = 거리_지도(창고, 자료['시험_정상'], 해상도); 불량_지도 = 거리_지도(창고, 자료['시험_불량'], 해상도)
    정상, 불량 = 이상_점수(정상_지도), 이상_점수(불량_지도); 적중 = 위치_적중(자료, 불량_지도)
    r = dict(제품=제품, 해상도=해상도, 걸린시간=time.time() - t, 자료=자료, 기준선=기준선, 정상=정상, 불량=불량, 적중=적중, 정상_지도=정상_지도, 불량_지도=불량_지도, 기준_지도=기준_지도)
    결과_보기(r, 그림)
    return r

def 결과_보기(r, 그림=True):
    정상, 불량, 기준선, 적중 = r['정상'], r['불량'], r['기준선'], r['적중']
    print(f"[{r['제품']} · 해상도 {r['해상도']}] 미검 {int((불량 <= 기준선).sum())} · 과검 {int((정상 > 기준선).sum())} · 구분력 {구분력(정상, 불량):.1f} · 위치 적중 {int(적중.sum())}/{len(적중)} ({r['걸린시간']:.0f}초)")
    if 그림:
        점수분포_그림(정상, 불량, 기준선, r['제품'] + ' — 점수 분포'); 잡은 = np.where(불량 > 기준선)[0]; 놓친 = np.where(불량 <= 기준선)[0]
        히트맵_보기(r['자료'], r['불량_지도'], (list(잡은[:2]) + list(놓친[:2])) if len(놓친) >= 2 else list(잡은[:4]), 눈금(r['기준_지도']))


print('도구 준비 완료 — 장치:', 장치)


## STEP 1 · 눈으로 먼저
기판(USB 충전 모듈) 사진 8장입니다. **불량은 몇 번일까요?** 30초 동안 직접 찾아보세요.


In [ ]:
제품 = "pcb4"     # 다른 제품: pcb1 · pcb3 · cashew · chewinggum · pipe_fryum · fryum · capsules · candle
자료 = 불러오기(제품)
문제 = 문제_고르기(자료)
보여주기(문제)


In [ ]:
# 정답 공개 — 빨간 선이 실제 결함 위치입니다
보여주기(문제, 정답=True)


## STEP 2 · 정상만 외우기
불량 사진은 한 장도 쓰지 않습니다. **정상 사진 120장**을 작은 조각으로 나눠 "기억 창고"에 넣습니다. (약 30초)


In [ ]:
기억할_장수 = 120     # 줄여 보세요: 30장만 있어도 될까요?
해상도 = 256          # 448로 올리면 작은 결함이 더 잘 보이지만 서너 배 느려집니다
t = time.time()
창고, 전체_조각 = 외우기(자료['기억'][:기억할_장수], 해상도)
print(f'정상 사진 {기억할_장수}장 → 조각 {전체_조각:,}개 → 그중 {len(창고):,}개를 기억했습니다 ({time.time() - t:.0f}초)')


## STEP 3 · 기준선 긋기
외울 때 쓰지 않은 **정상 사진 30장**의 이상 점수를 재 봅니다. "정상인데도 이 정도까지는 나온다" — 그 가운데 가장 높은 점수를 기준선으로 삼습니다.


In [ ]:
기준_지도 = 거리_지도(창고, 자료['기준'], 해상도)
기준_점수 = 이상_점수(기준_지도)
기준선 = float(기준_점수.max())
print(f'정상 30장의 이상 점수: {기준_점수.min():.2f} ~ {기준_점수.max():.2f}   →   기준선 = {기준선:.2f}')


## STEP 4 · 시험 — 처음 보는 사진 200장
정상 100장과 불량 100장을 채점합니다. 모델은 이 사진들을 **한 번도 본 적이 없습니다.** (약 1분)


In [ ]:
t = time.time()
정상_지도 = 거리_지도(창고, 자료['시험_정상'], 해상도)
불량_지도 = 거리_지도(창고, 자료['시험_불량'], 해상도)
정상_점수, 불량_점수 = 이상_점수(정상_지도), 이상_점수(불량_지도)
print(f'시험 사진 200장 채점 완료 ({time.time() - t:.0f}초)')
점수분포_그림(정상_점수, 불량_점수, 기준선)


In [ ]:
# 판정 결과 — 오전과 같은 표입니다. 놓침은 "미검", 헛경보는 "과검"이라고 부릅니다
결과 = 판정표(정상_점수, 불량_점수, 기준선)


### 쉬는 시간에 눌러 두는 칸
아래 칸은 **5분쯤** 걸립니다. 다른 기판(초음파 센서 모듈)을 **해상도를 올려서** 검사해 둡니다. ▶ 를 누르고 쉬러 가세요. 결과는 STEP 8에서 봅니다.


In [ ]:
기판_크게 = 한번에("pcb1", 해상도=448, 그림=False)
print('계산 완료 — 이 결과는 STEP 8에서 다시 봅니다')


## STEP 5 · 어디가 이상한가 — 근거 확인
모델이 "이상하다"고 본 곳을 색으로 칠합니다(히트맵). 이 데이터에는 **실제 결함 위치(정답)** 가 함께 들어 있어서, 모델이 짚은 곳이 맞는지 채점할 수 있습니다.

색은 **정상 사진에서는 한 번도 나오지 않은 수준**을 넘는 곳에만 칠합니다. 그래서 색이 옅거나 없는 사진도 있습니다.


In [ ]:
잡은_불량 = np.where(불량_점수 > 기준선)[0]
히트맵_보기(자료, 불량_지도, 잡은_불량[[0, len(잡은_불량) // 3, 2 * len(잡은_불량) // 3, -1]], 눈금(기준_지도))


In [ ]:
적중 = 위치_적중(자료, 불량_지도)
print(f'모델이 가장 이상하다고 짚은 지점이 실제 결함 위치에 든 사진: {int(적중.sum())} / {len(적중)}')


## STEP 6 · 기준선 손잡이 — 오전의 그 손잡이입니다
기준선을 올리면 과검이 줄고 미검이 늡니다. 내리면 반대입니다. 아래 **여유** 값을 바꾸고 다시 실행해 보세요.


In [ ]:
여유 = 1.0   #@param {type:"slider", min:0.9, max:1.2, step:0.01}
# 1.00 은 STEP 3에서 그은 기준선 그대로, 1.05 는 5% 올린 것입니다
판정표(정상_점수, 불량_점수, 기준선 * 여유, 그림=False)
print()
print('여유     미검   과검')
for k in [0.94, 0.97, 1.0, 1.03, 1.06, 1.1]:
    print(f'{k:.2f}   {int((불량_점수 <= 기준선 * k).sum()):5d}  {int((정상_점수 > 기준선 * k).sum()):5d}')


## STEP 7 · 틀린 사진 보기
숫자보다 중요한 것 — **무엇을 놓쳤는가.** 놓친 불량과 잘못 잡은 정상을 직접 봅니다.


In [ ]:
놓친_불량 = np.where(불량_점수 <= 기준선)[0]
print('놓친 불량:', len(놓친_불량), '장')
if len(놓친_불량): 히트맵_보기(자료, 불량_지도, 놓친_불량[:4], 눈금(기준_지도))


In [ ]:
과검_정상 = np.where(정상_점수 > 기준선)[0]
print('잘못 잡은 정상:', len(과검_정상), '장')
if len(과검_정상): 과검_보기(자료, 정상_지도, 과검_정상[:4], 눈금(기준_지도))


## STEP 8 · 한 줄만 바꿔 다른 제품
같은 방법을 다른 제품에 그대로 써 봅니다. 바꾸는 것은 **제품 이름 한 줄**뿐입니다. (제품마다 1분 남짓)

아래 그림은 **왼쪽 두 장이 잡은 불량, 오른쪽 두 장이 놓친 불량**입니다.


In [ ]:
껌 = 한번에("chewinggum")


In [ ]:
# 다른 기판(초음파 센서 모듈) — 이번에는 잘 될까요?
기판_작게 = 한번에("pcb1")


In [ ]:
# 쉬는 시간에 돌려 둔 결과 — 같은 기판, 같은 방법, 해상도만 256 → 448
결과_보기(기판_크게)


**읽는 법** — 같은 기판인데 사진을 크게 보았더니 놓친 불량이 크게 줄었습니다. 결함이 사진에서 몇 픽셀로 보이느냐가 성능을 갈랐습니다. **차이를 만든 것은 AI가 아니라 촬영 조건**입니다.

### 더 해 보기
- `한번에("capsules")` — 여러 알이 흩어져 놓인 캡슐. 해상도를 올려도 잘 풀리지 않습니다. 왜일까요?
- STEP 1의 `제품 = "pcb4"` 를 다른 이름으로 바꾸고 STEP 1부터 다시 실행
- STEP 2의 `기억할_장수 = 120` 을 30으로 줄여 보기


In [ ]:
# 정리 — 오늘 돌린 네 번의 결과를 한 표로 봅니다
for 이름, r in [(제품, dict(정상=정상_점수, 불량=불량_점수, 기준선=기준선, 적중=적중)), ('chewinggum', 껌), ('pcb1 256', 기판_작게), ('pcb1 448', 기판_크게)]:
    print(f"{이름:11s} 미검 {int((r['불량'] <= r['기준선']).sum()):3d} · 과검 {int((r['정상'] > r['기준선']).sum()):3d} · 구분력 {구분력(r['정상'], r['불량']):.1f} · 위치 적중 {int(r['적중'].sum())}/{len(r['적중'])}")


---
### 막혔을 때
1. **런타임 연결이 끊겼습니다** → 오른쪽 위 [다시 연결] → STEP 0부터 차례로 다시 실행합니다 (2~3분)
2. **이름이 정의되지 않았다는 오류(NameError)** → 위의 칸을 건너뛴 것입니다. STEP 0부터 차례로 실행합니다
3. **너무 느립니다** → STEP 4와 STEP 8은 1분 남짓, 쉬는 시간 칸은 5분쯤이 정상입니다. [런타임] → [런타임 유형 변경] → T4 GPU를 고르면 10초 안팎으로 줄어듭니다

### 데이터와 방법
- 데이터: VisA — Zou et al., "SPot-the-Difference Self-Supervised Pre-training for Anomaly Detection and Segmentation", ECCV 2022. Amazon, CC BY 4.0. 수업용으로 품목마다 350장을 뽑아 긴 변 448픽셀로 줄였습니다.
- 방법: PatchCore(Roth et al., CVPR 2022)를 수업용으로 줄인 구현입니다. 실제 설비에서는 조명과 지그, 해상도, 처리 속도에 맞춘 설계가 따로 필요합니다.
